# 🎯 2. Retrieval & Generation

Στο notebook 01 χτίσαμε ένα index. Τώρα θα δούμε:

1. Πώς να συναρμολογούμε **RAG chains** με το LCEL
2. Πώς να γράφουμε **prompts** που δίνουν αξιόπιστα αποτελέσματα
3. Πώς να εμφανίζουμε **πηγές** (source attribution)
4. Πώς να κάνουμε **streaming** την απάντηση
5. Πώς προσθέτουμε **chat history** σε ένα RAG (conversational RAG)

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
# if not os.environ.get("OPENAI_API_KEY"):
#     import getpass
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

## 2.1 Setup — knowledge base για όλο το notebook

Φτιάχνουμε ένα μεγαλύτερο knowledge base για την Datanous. Από εδώ και πέρα θα το χρησιμοποιούμε.

In [2]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
 
embedder = OpenAIEmbeddings(model=EMBED_MODEL)
 
# Datanous.ai knowledge base — defined here, reused throughout this notebook
datanous_kb = [
    Document(page_content="Datanous.ai is an AI services company founded in 2021 in Athens, Greece, with offices in London and Amsterdam.",
             metadata={"source": "datanous_overview.txt", "product": "company"}),
    Document(page_content="Datanous Insight is a RAG-powered knowledge management platform that indexes enterprise documents and answers queries with source citations.",
             metadata={"source": "datanous_products.txt", "product": "Insight"}),
    Document(page_content="Datanous Insight Starter costs 50 USD per month and supports up to 10,000 document pages with single-tenant isolation.",
             metadata={"source": "datanous_products.txt", "product": "Insight", "section": "pricing"}),
    Document(page_content="Datanous Insight Professional costs 350 USD per month, supports up to 100,000 pages, and includes multi-tenant row-level access control.",
             metadata={"source": "datanous_products.txt", "product": "Insight", "section": "pricing"}),
    Document(page_content="Datanous Insight Enterprise offers unlimited pages, a dedicated vector store, and a 99.9 percent uptime SLA at custom pricing.",
             metadata={"source": "datanous_products.txt", "product": "Insight", "section": "pricing"}),
    Document(page_content="Datanous Search combines BM25 lexical retrieval with dense semantic embeddings and applies cross-encoder reranking via Reciprocal Rank Fusion. Latency under 200 ms at the 95th percentile.",
             metadata={"source": "datanous_products.txt", "product": "Search"}),
    Document(page_content="Datanous Guard enforces factual grounding, detects prompt injection attacks, redacts PII (names, emails, IDs), and applies tenant isolation via tenant_id filtering.",
             metadata={"source": "datanous_products.txt", "product": "Guard"}),
    Document(page_content="Datanous Embed supports text-embedding-3-small (1536 dims) and text-embedding-3-large (3072 dims). Batch requests up to 2048 texts. Redis cache hit rate exceeds 60 percent.",
             metadata={"source": "datanous_products.txt", "product": "Embed"}),
    Document(page_content="Datanous.ai serves clients in banking, legal, healthcare, and media across Europe. All data encrypted with TLS 1.3 in transit and AES-256 at rest. GDPR-compliant.",
             metadata={"source": "datanous_faq.txt", "section": "privacy"}),
    Document(page_content="A law firm using Datanous Insight reduced document search time from 2.5 hours to 8 minutes and achieved 91 percent retrieval faithfulness.",
             metadata={"source": "datanous_case_studies.txt", "industry": "legal"}),
    Document(page_content="A retail bank deployed Datanous Insight Enterprise over 12,000 regulatory pages, reducing query resolution from 4 days to under 30 seconds.",
             metadata={"source": "datanous_case_studies.txt", "industry": "banking"}),
    Document(page_content="Datanous.ai evaluates every deployment on faithfulness, answer relevancy, context precision, and context recall using an LLM-as-judge pipeline on 200 held-out question-answer pairs.",
             metadata={"source": "datanous_blog_rag.txt", "section": "evaluation"}),
]
 
vectorstore = Chroma.from_documents(datanous_kb, embedding=embedder)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Knowledge base: {vectorstore._collection.count()} documents indexed")

Knowledge base: 12 documents indexed


## 2.2 Prompt engineering για RAG

Το prompt ενός RAG συστήματος έχει **τέσσερα κρίσιμα στοιχεία**:

1. **Persona** — ποιος είναι ο βοηθός
2. **Constraint** — να απαντά μόνο από το context
3. **Failure mode** — τι κάνει όταν δεν ξέρει
4. **Format** — πώς να γράψει την απάντηση

In [3]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """You are a helpful technical assistant for Datanous.ai
Answer the question using ONLY the information in the context below.
Rules:
1. If the answer is not in the context: "I do not have that information
2. Do not make up numbers, features, or product names.
3. Keep the answer concise and technically precise.
4. When relevant, mention the source product name.

Context:
{context}

Question: {question}

Answer:"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

print(f"Input variables: {rag_prompt.input_variables}")

Input variables: ['context', 'question']


# (RAG prompt defined in cell above)

<img src="images/lcel_pipeline.png" width="80%" style="border-radius:10px;margin:12px 0;"/>

***Εικ. 2.1** — Το LCEL pipeline: κάθε component είναι Runnable που συνδέεται με `|`. Η παράλληλη εκτέλεση (RunnableParallel) χτίζει το context και question ταυτόχρονα.*

Το **RunnableParallel** περνάει το ίδιο input σε πολλά Runnables και επιστρέφει dictionary με τα αποτελέσματα.

```python
{
    "context": "...σχετικα documents απο τον retriever",
    "question": "What is Datanous Insights and how does it cost?"
}
```

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    """Join retrieved documents into a single context string."""
    return "\n\n".join(
        f"[{d.metadata.get('product', d.metadata.get('section', 'general'))}] {d.page_content}"
        for d in docs
    )

llm = ChatOpenAI(model=LLM_MODEL, temperature=0.0)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("What is Datanous Insight and how much doest it cost?")
print("Q: What is Datanous Insight and how much doest it cost?")
print("A: ", answer)

Q: What is Datanous Insight and how much doest it cost?
A:  Datanous Insight is a RAG-powered knowledge management platform that indexes enterprise documents and answers queries with source citations. The cost of Datanous Insight Professional is 350 USD per month, while Datanous Insight Starter costs 50 USD per month.


### Πώς διαβάζεται το πρώτο dict;

```python
{"context": retriever | format_docs, "question": RunnablePassthrough()}
```

Είναι ένα **`RunnableParallel`** — εκτελεί κάθε key παράλληλα και επιστρέφει dict.

* `"context"`: παίρνει την είσοδο, την περνά στον retriever, μετά τη μορφοποιεί
* `"question"`: απλά περνά την είσοδο όπως είναι (`RunnablePassthrough`)

Έτσι το επόμενο βήμα (`rag_prompt`) δέχεται dict με τα `{context, question}` που χρειάζεται το template.

## 2.4 Source attribution — γιατί το LLM είπε αυτό;

Σε κάθε σοβαρό RAG πρέπει να μπορείς να πεις: «Η απάντηση βασίστηκε στις πηγές A, B, C».

Αυτό απαιτεί να **κρατήσουμε** τα retrieved documents παράλληλα με την απάντηση.

In [5]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def rag_with_citations(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)

    answer = (rag_prompt | llm | StrOutputParser()).invoke({"context": context, "question": question})

    print(f"Q: {question}")
    print(f"A: {answer}")
    print("Sources:")
    for d in docs:
        src = d.metadata.get("source", "unknown")
        product = d.metadata.get("product", d.metadata.get("section", ""))
        print(f"  [{product}] {src}: {d.page_content[:80]}...")

rag_with_citations("What does Datanous Guard protect against?")

Q: What does Datanous Guard protect against?
A: Datanous Guard protects against prompt injection attacks, enforces factual grounding, redacts PII (names, emails, IDs), and applies tenant isolation via tenant_id filtering.
Sources:
  [Guard] datanous_products.txt: Datanous Guard enforces factual grounding, detects prompt injection attacks, red...
  [privacy] datanous_faq.txt: Datanous.ai serves clients in banking, legal, healthcare, and media across Europ...
  [Insight] datanous_products.txt: Datanous Insight is a RAG-powered knowledge management platform that indexes ent...


## 2.5 Inline citations

Πιο εξελιγμένη παραλλαγή: ζητάμε από το LLM να **παραπέμπει σε αριθμημένες πηγές** μέσα στην απάντηση.

In [ ]:
def rag_with_score_filter(question: str, min_score: float = 0.55):
    """Retrieve docs, filter by relevance score, then generate."""
    scored = vectorstore.similarity_search_with_relevance_scores(question, k=5)
 
    relevant = [(doc, score) for doc, score in scored if score >= min_score]
    print(f"Q: {question}")
    print(f"Retrieved {len(scored)} docs, {len(relevant)} above threshold ({min_score}):")
    for doc, score in relevant:
        print(f"  [{score:.3f}] {doc.page_content[:80]}...")
 
    if not relevant:
        print("No sufficiently relevant documents found.")
        return
 
    context = "\n\n".join(doc.page_content for doc, _ in relevant)
    answer = (rag_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    print(f"\nA: {answer}")
 
rag_with_score_filter("What is the price of Datanous Insight Professional?")

Q: What is the price of Datanous Insight Professional?
Retrieved 5 docs, 1 above threshold (0.55):
  [0.698] Datanous Insight Professional costs 350 USD per month, supports up to 100,000 pa

A: The price of Datanous Insight Professional is 350 USD per month.


## 2.6 Structured output — JSON αντί για κείμενο

Συχνά θέλουμε η απάντηση να είναι structured (πχ για downstream API).
Με το `with_structured_output()` του LangChain, παίρνουμε type-safe Pydantic objects.

## 2.7 Streaming - απόκριση token-token

Για UX, οι χρήστες θέλουν να βλέπουν την απάντηση να εμφανίζεται **σταδιακά**, όχι να περιμένουν 5 δευτερόλεπτα.

Το LCEL υποστηρίζει streaming out-of-the-box — αρκεί να χρησιμοποιείς `.stream()` αντί για `.invoke()`.

## 2.8 Batch - πολλές ερωτήσεις παράλληλα

Όταν θες να τρέξεις πολλές ερωτήσεις (πχ για evaluation), `.batch()` τις τρέχει **παράλληλα**.

## 2.9 Conversational RAG - μνήμη συνομιλίας

Σε ένα chat, η νέα ερώτηση συχνά **εξαρτάται** από την προηγούμενη.

Παράδειγμα:

> User: What is Datanous Insight?  
> Bot: Datanous Insight is a RAG-powered knowledge management platform that indexes enterprise documents and answers queries with source citations.  
> User: **And how much does it cost for 100,000 pages?**

Η δεύτερη ερώτηση έχει νόημα **μόνο** αν ξέρουμε ότι το `it` αναφέρεται στο **Datanous Insight**.

Ένας naive retriever θα αναζητούσε κάτι σαν:

> And how much does it cost for 100,000 pages?

Το πρόβλημα είναι ότι η ερώτηση περιέχει αντωνυμία (`it`) και δεν είναι πλήρως αυτόνομη. Αν περάσει έτσι στον retriever, μπορεί να μη βρει το σωστό pricing document.

**Λύση:** δύο βήματα.

1. **Reformulate**: ένα LLM call ξαναγράφει την ερώτηση ως standalone.
2. **RAG**: ο retriever ψάχνει τη standalone version.

Τώρα συνδυάζουμε reformulate + retrieve + answer σε ένα chain.

## 2.10 Pitfalls στο RAG generation

Αυτά τα προβλήματα εμφανίζονται **ακόμα και με τέλειο retrieval**. Φύλαξε αυτή τη λίστα:

| Pitfall | Πώς εκδηλώνεται | Πώς το αντιμετωπίζεις |
|---|---|---|
| **Hallucination** | Το LLM προσθέτει «νούμερα» που δεν υπάρχουν | Auster prompt: «μόνο από context», temperature=0 |
| **Lost in the middle** | Αν δώσεις 10 docs, αγνοεί τα μεσαία | Re-rank ή fewer docs (notebook 06) |
| **Conflicting context** | 2 docs λένε διαφορετικά νούμερα | Πρόσθεσε στο prompt: «αν συγκρούονται, ζήτα διευκρίνιση» |
| **Stale data** | Index έχει παλιά τιμή → λάθος απάντηση | Re-index regularly + metadata `last_updated` |
| **Sycophancy** | Συμφωνεί με κάθε ισχυρισμό του χρήστη | Πρόσθεσε «μην συμφωνείς εκτός αν επιβεβαιώνεται από context» |
| **Context dilution** | Πολύ μεγάλο context → χάνει εστίαση | Compression / summarization (notebook 05) |

## 2.11 Άσκηση

**Άσκηση:** Φτιάξε ένα RAG chain που:
1. Δέχεται ερώτηση + προαιρετικό filter του product (πχ `"orbit_storage"`)
2. Κάνει retrieval με metadata filter όταν το product δίνεται
3. Επιστρέφει dict με `answer`, `sources` (λίστα από `(product, content)`)

## 📋 Συμπεράσματα

| # | Έννοια | Συνοπτικά |
|---|---|---|
| 1 | RAG prompt structure | persona + constraint + failure mode + format |
| 2 | LCEL | Composition με `\|`, κάθε βήμα είναι Runnable |
| 3 | `RunnableParallel` | Τρέχει πολλά keys ταυτόχρονα → dict output |
| 4 | `RunnablePassthrough` | Περνά την είσοδο όπως είναι |
| 5 | `.assign()` | Προσθέτει νέο key χωρίς να χάνεις τα υπόλοιπα |
| 6 | Source attribution | Κράτα τα retrieved docs μαζί με την απάντηση |
| 7 | Inline citations | Αριθμημένες πηγές [1], [2] στο prompt |
| 8 | Structured output | `with_structured_output(PydanticModel)` |
| 9 | `.stream()` | Token-by-token streaming για UX |
| 10 | `.batch()` | Παράλληλη εκτέλεση πολλών inputs |
| 11 | Conversational RAG | Reformulate σε standalone question πριν retrieval |
| 12 | Generation pitfalls | Hallucination, lost-in-the-middle, conflicting context |
| 13 | Επόμενο βήμα | Notebook 03 — Query Transformations |